# Case Study: Modeling NYC Citibike Demand

An end-to-end walkthrough on **real data**, the way a practitioner would approach
it: load a hosted dataset, *explore* it, fit two models, compare them, and
visualize what they learned.

We use **Citibike** — bike-share ride starts across New York City — hosted on the
[`seahorse-stpp`](https://huggingface.co/seahorse-stpp) Hugging Face organization.
Everything runs on CPU in a few minutes.

## Setup

Install Seahorse. In Colab (or any fresh runtime) this pulls the released package
from PyPI; locally it is a no-op if `seahorse` is already importable.

In [ ]:
import importlib.util

if importlib.util.find_spec("seahorse") is None:
    %pip install -q seahorse-stpp

import numpy as np
import matplotlib.pyplot as plt

from seahorse.data import load_dataset
from seahorse import PoissonGMM, DeepSTPP

## 1. Load the data

`load_dataset("citibike")` streams the train / val / test splits from the Hub and
caches them. Each record is one sequence with aligned `times` (hours into the day)
and `locations` (longitude, latitude).

In [ ]:
splits = load_dataset("citibike")
train, val, test = splits["train"], splits["val"], splits["test"]

print("sequences per split:", {k: len(v) for k, v in splits.items()})
ex = train[0]
print(f"first sequence: {len(ex['times'])} events")
print("times[:3]     =", np.round(ex["times"][:3], 3).tolist())
print("locations[:2] =", np.round(ex["locations"][:2], 3).tolist())

## 2. Explore the data

Before modeling, get a feel for it — a consultant's first questions: how many
events per sequence, and *when* do rides happen?

In [ ]:
counts = np.array([len(s["times"]) for s in train])
all_times = np.concatenate([np.asarray(s["times"], float) for s in train])
all_locs = np.concatenate([np.asarray(s["locations"], float) for s in train])

print(f"{len(train)} sequences, {counts.sum():,} events")
print(f"events / sequence: min {counts.min()}, median {int(np.median(counts))}, max {counts.max()}")
print(f"time span: [{all_times.min():.2f}, {all_times.max():.2f}] hours")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].hist(counts, bins=30, color="#0284c7")
ax[0].set_title("Events per sequence"); ax[0].set_xlabel("events")
ax[1].hist(all_times, bins=48, color="#0284c7")
ax[1].set_title("When rides start"); ax[1].set_xlabel("hour of day")
plt.tight_layout(); plt.show()

### And *where*?

The locations are real NYC coordinates, so plotting them draws the city. On the
left, a sample of individual ride starts; on the right, their density — the
spatial pattern a model has to capture.

In [ ]:
rng = np.random.default_rng(0)
idx = rng.choice(len(all_locs), size=min(20000, len(all_locs)), replace=False)
sample = all_locs[idx]

fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
ax[0].scatter(sample[:, 0], sample[:, 1], s=2, alpha=0.25, color="#0284c7")
ax[0].set_title("Ride starts (20k sample)")
hb = ax[1].hexbin(all_locs[:, 0], all_locs[:, 1], gridsize=60, cmap="magma", bins="log")
ax[1].set_title("Ride density (log scale)")
fig.colorbar(hb, ax=ax[1], label="log count")
for a in ax:
    a.set_xlabel("longitude"); a.set_ylabel("latitude"); a.set_aspect("equal")
plt.tight_layout(); plt.show()

## 3. A quick CPU slice

Fitting on every full-length sequence is heavy for a CPU tutorial, so we take a
small subsample and truncate each sequence to its first 40 events — enough to see
the mechanics quickly. For real experiments, use more sequences, full-length data,
more epochs, and a GPU (or the benchmark CLI).

In [ ]:
def prepare(seqs, n_seq, k=40):
    """Small subsample, each sequence truncated to its first k events."""
    return [{"times": s["times"][:k], "locations": s["locations"][:k]} for s in seqs[:n_seq]]

train_s = prepare(train, 60)
val_s = prepare(val, 20)
test_s = prepare(test, 20)
print(f"training slice: {len(train_s)} sequences x {len(train_s[0]['times'])} events")

## 4. Fit two models

The same `fit()` call trains both — `PoissonGMM`, a fast parametric baseline, and
`DeepSTPP`, a neural model. Seahorse applies identical splits, normalization, and
metric to each, so their scores are comparable by construction.

In [ ]:
%%capture
baseline = PoissonGMM(device="cpu", seed=0)
neural = DeepSTPP(device="cpu", seed=0)

baseline.fit(train_s, val_s, test_s, epochs=3, batch_size=16, dataset_id="citibike")
neural.fit(train_s, val_s, test_s, epochs=3, batch_size=16, dataset_id="citibike")

## 5. Compare head-to-head

Both are exact-likelihood models, so their test NLL is directly comparable — lower
is better.

In [ ]:
scores = {
    "PoissonGMM": float(baseline.evaluate(test_s)["test_nll"]),
    "DeepSTPP": float(neural.evaluate(test_s)["test_nll"]),
}
print("test NLL:", {k: round(v, 3) for k, v in scores.items()})

fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(list(scores), list(scores.values()), color=["#94a3b8", "#0284c7"])
ax.set_ylabel("test NLL (lower is better)")
ax.set_title("Head-to-head")
plt.tight_layout(); plt.show()

## 6. Where does the model expect the next ride?

`predict_next` samples the next event for held-out contexts. Overlaying the neural
model's samples (blue) against the true next location (star) shows *where* it
places its predictions — returned in the original NYC coordinates.

In [ ]:
samples = neural.predict_next(test_s[:5], n_samples=32)
ctx = 0
pred = np.asarray(samples["next_locations"])[ctx]
true = np.asarray(samples["true_next_locations"])[ctx]

fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.scatter(sample[:, 0], sample[:, 1], s=2, alpha=0.08, color="#94a3b8", label="all ride starts")
ax.scatter(pred[:, 0], pred[:, 1], s=20, alpha=0.75, color="#0284c7", label="DeepSTPP samples")
ax.scatter([true[0]], [true[1]], marker="*", s=280, color="#f59e0b",
           edgecolor="black", linewidth=0.6, label="true next", zorder=5)
ax.set_title("Where does the model expect the next ride?")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_aspect("equal"); ax.legend(loc="best", fontsize=8)
plt.tight_layout(); plt.show()

## What to try next

- **Swap the dataset** — any of the 13 in the catalog loads by short name:
  `load_dataset("covid")`, `load_dataset("earthquakes")`, ...
- **Scale up** — more sequences, full-length data, more epochs, a GPU.
- **Add more models** — compare presets and seeds with the benchmark CLI.
- **Richer surfaces** — `pip install plotly` unlocks
  `model.plot_intensity(seq)` for an interactive intensity surface.